# Create the dataset for computation

In [1]:
import datasets
import pandas as pd
import numpy as np

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
    compute_neighbors_and_distances,
    compute_neighbor_weighted_prices,
)

In [2]:
txt_only = False
include_embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{include_embeddings}"

In [3]:
window = 28
mod = 4
max_periods = 54

ds_dict = datasets.load_dataset("DoubleML/amzn_toys_monthly_diffs_ffill_fixed_splits_buybox")

ds_val = ds_dict['validation']
ds_train = ds_dict['train']

columns = ['ASIN',
           'SALES_RANK',
           'PRICE',
           'BUYBOX_PRICE',
           'text',
           'date',
           'window',
           'REVIEW_COUNT',
           'RATING',
           'New Offer Count: Current',
           'Count of retrieved live offers: New, FBA',
           'Count of retrieved live offers: New, FBM',
           'Lightning Deals: Upcoming Deal',
           'Buy Box: Is FBA',
           'subcat_aggregated']

df_val = ds_val.select_columns(columns).to_pandas()
df_val = df_val[df_val["window"] == window].dropna()
df_val["date_t"] = df_val["date"].astype("category").cat.codes
df_val = df_val[df_val["date_t"] <= max_periods]
df_val = df_val[df_val["date_t"] % mod == 0]

df_train = ds_train.select_columns(columns).to_pandas()
df_train = df_train[df_train["window"] == window].dropna()
df_train["date_t"] = df_train["date"].astype("category").cat.codes
df_train = df_train[df_train["date_t"] <= max_periods]
df_train = df_train[df_train["date_t"] % mod == 0]


df_train_val = pd.concat([df_train, df_val], axis=0).reset_index()

# Dataset contains data with different averaged values (by window size).
# Predictions / Embeddings only for one specific window size

num_val = len(df_val)
num_train = len(df_train)

print(f"Number of train samples: {num_train}")
print(f"Number of val samples: {num_val}")

# Create dummy variables for the subcategories
dummy_subcat = pd.get_dummies(df_train_val["subcat_aggregated"]) * 1
dummy_subcat_names = list(dummy_subcat.columns)
dummy_time = pd.get_dummies(df_train_val["date"]) * 1
dummy_time_names = list(dummy_time.columns)

df_train_val = pd.concat([df_train_val, dummy_subcat, dummy_time], axis=1)

df_full = pd.concat([df_train_val, dummy_subcat], axis=1)
df_full.set_index(["ASIN", "date"], inplace=True)

print(f"df_full shape: {df_full.shape}")
print(df_full.columns)

Number of train samples: 41945
Number of val samples: 41832
df_full shape: (83777, 51)
Index(['index', 'SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       'Count of retrieved live offers: New, FBM',
       'Lightning Deals: Upcoming Deal', 'Buy Box: Is FBA',
       'subcat_aggregated', 'date_t', 'Airplanes', 'Cars & Race Cars',
       'Motor Vehicles', 'Race Tracks', 'Residual', 'Skateboards', 'Tractors',
       'Train Sets', 'Trains & Trams', 'Trucks', 'Vehicle Playsets',
       '2023-01-30', '2023-02-27', '2023-03-27', '2023-04-24', '2023-05-22',
       '2023-06-19', '2023-07-17', '2023-08-14', '2023-09-11', '2023-10-09',
       '2023-11-06', '2023-12-04', '2024-01-01', '2024-01-29', 'Airplanes',
       'Cars & Race Cars', 'Motor Vehicles', 'Race Tracks', 'Residual',
       'Skateboards', 'Tractors', 'Train Sets', 'Trains & Trams', 'Trucks',
       'Vehicle Play

In [4]:
results_256_time_independent, configs_time_independent = load_pred_and_emb(
        embedding_size=256,
        txt_only=txt_only,
        lag_type="time_independent",
        config_path="utils/paths_config.yaml",
        get_lag2_for_diff=False,
    )


results_256_lag1, configs_lag1 = load_pred_and_emb(
        embedding_size=256,
        txt_only=txt_only,
        lag_type="lag1",
        config_path="utils/paths_config.yaml",
        get_lag2_for_diff=False,
    )

if not txt_only:
    results_256_lag2, configs_lag2 = load_pred_and_emb(
            embedding_size=256,
            txt_only=txt_only,
            lag_type="lag1",
            config_path="utils/paths_config.yaml",
            get_lag2_for_diff=False,
        )

[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (41832, 258)
  - val_embeddings_diff:     (38041, 130)
  - train_embeddings_levl:   (41945, 258)
  - val_predictions_levl:    (41832, 4)
  - val_predictions_diff:    (38041, 4)
  - train_predictions_levl:  (41945, 4)
  - train_predictions_diff:  (38146, 4)
[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (38203, 258)
  - val_embeddings_diff:     (38041, 130)
  - train_embeddings_levl:   (38309, 258)
  - val_predictions_levl:    (38203, 4)
  - val_predictions_diff:    (38041, 4)
  - train_predictions_levl:  (38309, 4)
  - train_predictions_diff:  (38146, 4)
[INFO] Shapes for embeddings & predictions (level/diff) loaded:
  - val_embeddings_levl:     (38203, 258)
  - val_embeddings_diff:     (38041, 130)
  - train_embeddings_levl:   (38309, 258)
  - val_predictions_levl:    (38203, 4)
  - val_predictions_diff:    (38041, 4)
  - train_predictions_levl:  (

## Create Embeddings, Similarites and PCA based on time independent embeddings

In [5]:
pred_and_emb = results_256_time_independent
embeddings = center_and_norm(pred_and_emb["embeddings"][2], pred_and_emb["embeddings"][0])
print(f"Embedding shape: {embeddings.shape}")
_, cluster_centroids = get_cluster(embeddings, n_clusters=5, n_init=200)
pca_results = get_pca(embeddings, n_components=5)
df_pca = pd.DataFrame(pca_results, columns=[f"pca_{i}" for i in range(pca_results.shape[1])])
df_sim = get_similarities(embeddings, cluster_centroids)

print(f"pca columns: {df_pca.columns}")
print(f"sim columns: {df_sim.columns}")

# join to dataframe
df_full = df_full.join(df_pca)
df_full = df_full.join(df_sim)

if include_embeddings:
    df_full = df_full.join(embeddings.add_prefix("emb_"))

Embedding shape: (83777, 256)
pca columns: Index(['pca_0', 'pca_1', 'pca_2', 'pca_3', 'pca_4'], dtype='object')
sim columns: Index(['similarity_cluster_0', 'similarity_cluster_1', 'similarity_cluster_2',
       'similarity_cluster_3', 'similarity_cluster_4'],
      dtype='object')


In [6]:
# ## add treatment_features and outcome_features

# t_f = [f"treatment_feature_{i}" for i in range(5)]
# o_f = [f"outcome_feature_{i}" for i in range(5)]

# treat_features_val = results_256_time_independent["embeddings"][0][["ASIN"] + t_f]
# treat_features_trn = results_256_time_independent["embeddings"][2][["ASIN"] + t_f]
# outcome_features_val = results_256_time_independent["embeddings"][0][["ASIN"] + o_f]
# outcome_features_trn = results_256_time_independent["embeddings"][2][["ASIN"] + o_f]

# treat_features = pd.concat([treat_features_trn, treat_features_val], axis=0)
# outcome_features = pd.concat([outcome_features_trn, outcome_features_val], axis=0)

### Generate weighted substitute price feature

In [7]:
df_neighbor_asins_by_date, df_distance_df_by_date = compute_neighbors_and_distances(embeddings, n_neighbors=5)

df_full = df_full.join(df_neighbor_asins_by_date)
df_full = df_full.join(df_distance_df_by_date)

print(f"df_full shape after joining embeddings and neighbors: {df_full.shape}")

Combined neighbor ASINs shape: (83777, 5)
Combined neighbor distances shape: (83777, 5)
df_full shape after joining embeddings and neighbors: (83777, 327)


In [8]:
df_full.head()

index  SALES_RANK     PRICE  BUYBOX_PRICE  \
ASIN       date                                                    
1947335162 2023-01-30      1  -12.747395  3.400864      3.555062   
           2023-02-27      9  -12.600586  3.404632      3.555062   
           2023-03-27     17  -12.515369  3.427422      3.633271   
           2023-04-24     25  -12.576837  3.400864      3.582963   
           2023-05-22     33  -12.688608  3.400864      3.587493   

                                                                    text  \
ASIN       date                                                            
1947335162 2023-01-30  Catalyst Game Labs Battletech: Technical Reado...   
           2023-02-27  Catalyst Game Labs Battletech: Technical Reado...   
           2023-03-27  Catalyst Game Labs Battletech: Technical Reado...   
           2023-04-24  Catalyst Game Labs Battletech: Technical Reado...   
           2023-05-22  Catalyst Game Labs Battletech: Technical Reado...   

                       window  REVIEW_COUNT    RATING  \
ASIN       date                                         
1947335162 2023-01-30    28.0     62.000000  4.700000   
           2023-02-27    28.0     62.000000  4.700000   
           2023-03-27    28.0     62.035714  4.678571   
           2023-04-24    28.0     63.607143  4.700000   
           2023-05-22    28.0     64.000000  4.700000   

                       New Offer Count: Current  \
ASIN       date                                   
1947335162 2023-01-30                        10   
           2023-02-27                        10   
           2023-03-27                        10   
           2023-04-24                        10   
           2023-05-22                        10   

                       Count of retrieved live offers: New, FBA  ...  \
ASIN       date                                                  ...   
1947335162 2023-01-30                                         2  ...   
           2023-02-27                                         2  ...   
           2023-03-27                                         2  ...   
           2023-04-24                                         2  ...   
           2023-05-22                                         2  ...   

                       neighbor_asin_1  neighbor_asin_2  neighbor_asin_3  \
ASIN       date                                                            
1947335162 2023-01-30       B09QH7Z7WF       B08X45DJ15       B09XCR1DFQ   
           2023-02-27       B09QH7Z7WF       B08X45DJ15       B09XCR1DFQ   
           2023-03-27       B09QH7Z7WF       B08X45DJ15       B09XCR1DFQ   
           2023-04-24       B01L4GYXCY       B09QH7Z7WF       B08X45DJ15   
           2023-05-22       B01L4GYXCY       B08X45DJ15       B09QH7Z7WF   

                      neighbor_asin_4  neighbor_asin_5  neighbor_distance_1  \
ASIN       date                                                               
1947335162 2023-01-30      B09QH7C9YH       B01LFEWHKA             0.194367   
           2023-02-27      B09QH7C9YH       B01LFEWHKA             0.195522   
           2023-03-27      B01L4GYXCY       B09QH7C9YH             0.195522   
           2023-04-24      B09XCR1DFQ       B09QH7C9YH             0.191243   
           2023-05-22      B09QH7C9YH       B09XCR1DFQ             0.191243   

                       neighbor_distance_2  neighbor_distance_3  \
ASIN       date                                                   
1947335162 2023-01-30             0.196050             0.199138   
           2023-02-27             0.195735             0.198824   
           2023-03-27             0.195735             0.198824   
           2023-04-24             0.195522             0.195735   
           2023-05-22             0.192912             0.195522   

                       neighbor_distance_4  neighbor_distance_5  
ASIN       date                                                  
1947335162 2023-01-30             0.202013             0.2071

In [9]:
[x for x in df_full.columns if "outcome" in x.lower()]

[]

In [10]:
weighted_substitute_price = compute_neighbor_weighted_prices(df_full, 'BUYBOX_PRICE')
df_full = df_full.join(weighted_substitute_price)
df_full.loc[:, ['BUYBOX_PRICE', 'weighted_substitute_price']].head()

Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN columns.
(5,)
Found 5 neighbor ASIN column

BUYBOX_PRICE  weighted_substitute_price
ASIN       date                                               
1947335162 2023-01-30      3.555062                   3.580552
           2023-02-27      3.555062                   3.603493
           2023-03-27      3.633271                   3.717575
           2023-04-24      3.582963                   3.619921
           2023-05-22      3.587493                   3.644166

In [12]:
df_full_val = df_full.iloc[num_train:, :]
df_full_train = df_full.iloc[:num_train, :]

print(f"df_full shape: {df_full.shape}")
print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")
print(df_full.columns)

df_full shape: (83777, 328)
df_full_train shape: (41945, 328)
df_full_val shape: (41832, 328)
Index(['index', 'SALES_RANK', 'PRICE', 'BUYBOX_PRICE', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       ...
       'neighbor_asin_2', 'neighbor_asin_3', 'neighbor_asin_4',
       'neighbor_asin_5', 'neighbor_distance_1', 'neighbor_distance_2',
       'neighbor_distance_3', 'neighbor_distance_4', 'neighbor_distance_5',
       'weighted_substitute_price'],
      dtype='object', length=328)


In [15]:
print(df_full_val.isna().sum().sort_values(ascending=False).head(20))

index                                       0
SALES_RANK                                  0
PRICE                                       0
BUYBOX_PRICE                                0
text                                        0
window                                      0
REVIEW_COUNT                                0
RATING                                      0
New Offer Count: Current                    0
Count of retrieved live offers: New, FBA    0
Count of retrieved live offers: New, FBM    0
Lightning Deals: Upcoming Deal              0
Buy Box: Is FBA                             0
subcat_aggregated                           0
date_t                                      0
Airplanes                                   0
Cars & Race Cars                            0
Motor Vehicles                              0
Race Tracks                                 0
Residual                                    0
dtype: int64


In [16]:
print(df_full_train.isna().sum().sort_values(ascending=False).head(20))

index                                       0
SALES_RANK                                  0
PRICE                                       0
BUYBOX_PRICE                                0
text                                        0
window                                      0
REVIEW_COUNT                                0
RATING                                      0
New Offer Count: Current                    0
Count of retrieved live offers: New, FBA    0
Count of retrieved live offers: New, FBM    0
Lightning Deals: Upcoming Deal              0
Buy Box: Is FBA                             0
subcat_aggregated                           0
date_t                                      0
Airplanes                                   0
Cars & Race Cars                            0
Motor Vehicles                              0
Race Tracks                                 0
Residual                                    0
dtype: int64


In [17]:
df_full_val = df_full_val.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"})
for i in range(1, 3):
    df_full_val[f"Q_t-{i}"] = df_full_val.groupby("ASIN")["Q_t"].shift(i)
    df_full_val[f"P_bb_t-{i}"] = df_full_val.groupby("ASIN")["P_bb_t"].shift(i)
    df_full_val[f"REVIEW_COUNT_t-{i}"] = df_full_val["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_val[f"RATING_t-{i}"] = df_full_val["RATING"].groupby("ASIN").shift(i)

df_full_val["Delta_Q_t"] = df_full_val["Q_t"] - df_full_val["Q_t-1"]
df_full_val["Delta_P_bb_t"] = df_full_val["P_bb_t"] - df_full_val["P_bb_t-1"]


df_full_train.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}, inplace=True)
for i in range(1, 3):
    df_full_train[f"Q_t-{i}"] = df_full_train["Q_t"].groupby("ASIN").shift(i)
    df_full_train[f"P_bb_t-{i}"] = df_full_train["P_bb_t"].groupby("ASIN").shift(i)
    df_full_train[f"REVIEW_COUNT_t-{i}"] = df_full_train["REVIEW_COUNT"].groupby("ASIN").shift(i)
    df_full_train[f"RATING_t-{i}"] = df_full_train["RATING"].groupby("ASIN").shift(i)
    
df_full_train["Delta_Q_t"] = df_full_train["Q_t"] - df_full_train["Q_t-1"]
df_full_train["Delta_P_bb_t"] = df_full_train["P_bb_t"] - df_full_train["P_bb_t-1"]


val_predictions_levl = pred_and_emb["predictions"][0]
val_predictions_diff = pred_and_emb["predictions"][1].rename(columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"})
train_predictions_levl = pred_and_emb["predictions"][2]
train_predictions_diff = pred_and_emb["predictions"][3].rename(columns={"pred_ml_l": "pred_ml_l_diff", "pred_ml_m": "pred_ml_m_diff"})

df_full_val = df_full_val.join(val_predictions_levl.set_index(["ASIN", "date"]))
df_full_train = df_full_train.join(train_predictions_levl.set_index(["ASIN", "date"]))
df_full_val = df_full_val.join(val_predictions_diff.set_index(["ASIN", "date"]))
df_full_train = df_full_train.join(train_predictions_diff.set_index(["ASIN", "date"]))

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")

print(f"df_full_train columns: {df_full_train.columns}")
print(f"df_full_val columns: {df_full_val.columns}")

C:\Users\Work\AppData\Local\Temp\ipykernel_22120\4146551084.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_full_train.rename(columns={"SALES_RANK": "Q_t", "BUYBOX_PRICE": "P_bb_t"}, inplace=True)
C:\Users\Work\AppData\Local\Temp\ipykernel_22120\4146551084.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_full_train[f"Q_t-{i}"] = df_full_train["Q_t"].groupby("ASIN").shift(i)
C:\Users\Work\AppData\Local\Temp\ipykernel_22120\4146551084.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row

df_full_train shape: (41945, 342)
df_full_val shape: (41832, 342)
df_full_train columns: Index(['index', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT',
       'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       ...
       'Q_t-2', 'P_bb_t-2', 'REVIEW_COUNT_t-2', 'RATING_t-2', 'Delta_Q_t',
       'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff'],
      dtype='object', length=342)
df_full_val columns: Index(['index', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT',
       'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       ...
       'Q_t-2', 'P_bb_t-2', 'REVIEW_COUNT_t-2', 'RATING_t-2', 'Delta_Q_t',
       'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff'],
      dtype='object', length=342)


### Add Lag 1 predictions to the dataset

In [18]:
val_predictions_levl_lag1 = results_256_lag1["predictions"][0].rename(columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"})
val_predictions_diff_lag1 = results_256_lag1["predictions"][1].rename(columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"})
train_predictions_levl_lag1 = results_256_lag1["predictions"][2].rename(columns={"pred_ml_l": "pred_ml_l_lag_1", "pred_ml_m": "pred_ml_m_lag_1"})
train_predictions_diff_lag1 = results_256_lag1["predictions"][3].rename(columns={"pred_ml_l": "pred_ml_l_diff_lag_1", "pred_ml_m": "pred_ml_m_diff_lag_1"})

df_full_val = df_full_val.join(val_predictions_levl_lag1.set_index(["ASIN", "date"]))
df_full_train = df_full_train.join(train_predictions_levl_lag1.set_index(["ASIN", "date"]))
df_full_val = df_full_val.join(val_predictions_diff_lag1.set_index(["ASIN", "date"]))
df_full_train = df_full_train.join(train_predictions_diff_lag1.set_index(["ASIN", "date"]))

print(f"df_full_train shape: {df_full_train.shape}")
print(f"df_full_val shape: {df_full_val.shape}")

print(f"df_full_train columns: {df_full_train.columns}")
print(f"df_full_val columns: {df_full_val.columns}")

df_full_train shape: (41945, 346)
df_full_val shape: (41832, 346)
df_full_train columns: Index(['index', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT',
       'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=346)
df_full_val columns: Index(['index', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window', 'REVIEW_COUNT',
       'RATING', 'New Offer Count: Current',
       'Count of retrieved live offers: New, FBA',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=346)


### Add Lag 2 predictions to the dataset

In [14]:
if False: # not txt_only:
    val_predictions_diff_lag2 = results_256_lag2["predictions"][1].rename(columns={"pred_ml_l": "pred_ml_l_diff_lag_2", "pred_ml_m": "pred_ml_m_diff_lag_2"})
    train_predictions_diff_lag2 = results_256_lag2["predictions"][3].rename(columns={"pred_ml_l": "pred_ml_l_diff_lag_2", "pred_ml_m": "pred_ml_m_diff_lag_2"})

    df_full_val = df_full_val.join(val_predictions_diff_lag2.set_index(["ASIN", "date"]))
    df_full_train = df_full_train.join(train_predictions_diff_lag2.set_index(["ASIN", "date"]))

    print(f"df_full_train shape: {df_full_train.shape}")
    print(f"df_full_val shape: {df_full_val.shape}")

    print(f"df_full_train columns: {df_full_train.columns}")
    print(f"df_full_val columns: {df_full_val.columns}")
    print(configs_lag2)
else:
    print("Skipping lag2")

Skipping lag2


In [19]:
df_full_val.reset_index()["date"].dropna().unique()

array(['2023-01-30', '2023-02-27', '2023-03-27', '2023-04-24',
       '2023-05-22', '2023-06-19', '2023-07-17', '2023-08-14',
       '2023-09-11', '2023-10-09', '2023-11-06', '2023-12-04',
       '2024-01-01', '2024-01-29'], dtype=object)

write to csv 

In [20]:
drop_cols=[
    'Q_t-2',
    'P_bb_t-2',
    'REVIEW_COUNT_t-2',
    'RATING_t-2',
    'pred_ml_l_lag_1',
    'pred_ml_m_lag_1',
]

keep_cols = [c for c in df_full_val.columns if c not in drop_cols]


In [21]:
print(df_full_val[keep_cols].isna().sum().sort_values(ascending=False).head(20))
print("\n\n")
print(df_full_train[keep_cols].isna().sum().sort_values(ascending=False).head(20))

pred_ml_m_diff_lag_1                        3791
pred_ml_l_diff                              3791
pred_ml_m_diff                              3791
pred_ml_l_diff_lag_1                        3791
Q_t-1                                       3775
P_bb_t-1                                    3775
REVIEW_COUNT_t-1                            3775
RATING_t-1                                  3775
Delta_Q_t                                   3775
Delta_P_bb_t                                3775
Count of retrieved live offers: New, FBM       0
emb_248                                        0
emb_249                                        0
emb_250                                        0
emb_251                                        0
emb_252                                        0
emb_253                                        0
emb_254                                        0
emb_255                                        0
neighbor_asin_1                                0
dtype: int64



pred

In [17]:
# rename weighted_substitute_price_lagged to weighted_substitute_price in both ds

df_full_val = df_full_val.rename(columns={"weighted_substitute_price_lagged": "weighted_substitute_price"})
df_full_train = df_full_train.rename(columns={"weighted_substitute_price_lagged": "weighted_substitute_price"})

In [ ]:
list(df_full_val.columns)

In [21]:
df_full_train.to_csv(f"predictions/datasets/{dataframe_name}_train.csv")
df_full_val.to_csv(f"predictions/datasets/{dataframe_name}_val.csv")